# Machine Learning Assignment 2: Dry Bean classification

### 1. Importing Libraries

In [2]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from common import FEATURES, MODEL_FILES, TARGET, evaluate_model

SEED = 42

### 2. Split the Dataset

In [3]:
data = pd.read_csv("data/dry_bean_raw.csv").drop_duplicates().reset_index(drop=True)
train, test = train_test_split(
    data, test_size=0.2, stratify=data[TARGET], random_state=SEED
)

test.to_csv("test_data.csv", index=False)
test = pd.read_csv("test_data.csv")

print(f"Training rows: {len(train):,} | Test rows: {len(test):,}")

Training rows: 10,834 | Test rows: 2,709


### 3. Defining Five Classifiers (Logistic Regression, Decision Tree, KNN, Naive Bayes, SVM)

In [4]:
def make_pipeline(estimator, scale=True):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    return Pipeline(steps + [("classifier", estimator)])


models = {
    "Logistic Regression": make_pipeline(LogisticRegression(C=1.0, max_iter=2500)),
    "Decision Tree": make_pipeline(
        DecisionTreeClassifier(max_depth=10, random_state=SEED), scale=False
    ),
    "KNN": make_pipeline(KNeighborsClassifier(n_neighbors=15, weights="distance")),
    "Naive Bayes": make_pipeline(GaussianNB(var_smoothing=1e-11)),
    "Support Vector Machine": make_pipeline(
        SVC(C=1.0, gamma=0.05, probability=True, random_state=SEED)
    ),
}

### 4. Model Training

In [5]:
model_dir = Path("model")
model_dir.mkdir(exist_ok=True)
results = []

for name, model in models.items():
    model.fit(train[FEATURES], train[TARGET])
    joblib.dump(model, model_dir / MODEL_FILES[name], compress=3)
    scores = evaluate_model(model, test[FEATURES], test[TARGET])
    results.append({"Model": name, **scores})

pd.DataFrame(results).set_index("Model").round(4)

,Accuracy,AUC,Precision,Recall,F1,MCC
Model,,,,,,
Logistic Regression,0.9192,0.9947,0.9307,0.9300,0.9302,0.9023
Decision Tree,0.9037,0.9693,0.9191,0.9176,0.9182,0.8835
KNN,0.9155,0.9899,0.9302,0.9265,0.9279,0.8979
Naive Bayes,0.8970,0.9911,0.9081,0.9082,0.9072,0.8762
Support Vector Machine,0.9210,0.9948,0.9345,0.9318,0.9330,0.9045
